# 03 — Baseline Strategy Comparison

**DiscoveryRank Phase 1 — Ranking & Evaluation**

This notebook evaluates three simple, non-ML baseline ranking strategies across the four DiscoveryRank evaluation dimensions:
- **Relevance** — do the ranked items match what the user actually engaged with?
- **Freshness** — how recently uploaded are the recommended items?
- **Diversity** — how varied is the content pool (authors, tags)?
- **Repetition Risk** — does the strategy create a repetitive feed?

**Data source:** `outputs/pipeline_sample.csv` (built by notebook 01, from `log_random_4_22_to_5_08_1k.csv`).  
**Scope:** 50 users with deep session histories, evaluated across all their sessions.

In [1]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('../src'))
import ranking_strategies
import eval_metrics

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

Imports OK


## 1. Load Pipeline Sample

We load the prepared pipeline output from notebook 01. This already contains:
`y_relevant`, `implicit_completion_ratio`, `item_age_days`, `session_id`, `author_id`, `tag`.

In [2]:
sample_path = '../outputs/pipeline_sample.csv'
df = pd.read_csv(sample_path)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Columns:', df.columns.tolist())

Loaded: 43,028 rows × 39 columns
Columns: ['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click', 'is_like', 'is_follow', 'is_comment', 'is_forward', 'is_hate', 'long_view', 'play_time_ms', 'duration_ms', 'profile_stay_time', 'comment_stay_time', 'is_profile_enter', 'is_rand', 'tab', 'author_id', 'video_type', 'upload_dt', 'upload_type', 'visible_status', 'video_duration', 'server_width', 'server_height', 'music_id', 'music_type', 'tag', 'session_id', 'explicit_positive_any', 'explicit_negative', 'implicit_completion_ratio', 'y_relevant', 'upload_dt_parsed', 'upload_time_ms', 'item_age_ms', 'item_age_days']


## 2. Subset Selection

Select users with at least 5 interactions and take up to 50 users.
This gives us a manageable but representative set of sessions to compare across.

In [3]:
# Keep only users with >= 5 interactions (enough to form a meaningful ranked set)
user_counts = df['user_id'].value_counts()
eligible_users = user_counts[user_counts >= 5].index.tolist()
selected_users = eligible_users[:50]

subset = df[df['user_id'].isin(selected_users)].copy()
print(f'Selected {len(selected_users)} users, {subset.shape[0]:,} interaction rows')
print(f'Unique sessions in subset: {subset["session_id"].nunique()}')

Selected 50 users, 12,162 interaction rows


Unique sessions in subset: 3245


## 3. Run All 3 Strategies per Session

For each (user, session) pair, we:
1. Apply all 3 ranking strategies to get a ranked DataFrame
2. Join back any feature columns needed by the metric functions
3. Compute all 4 metric groups via `eval_metrics.score_all_metrics`

**Note:** `deja_vu_rate` is computed per session using the user's prior session video IDs as history.

In [4]:
STRATEGY_FUNCS = {
    'popularity_based': ranking_strategies.popularity_based,
    'freshness_boosted': ranking_strategies.freshness_boosted,
    'diversity_aware_rerank': ranking_strategies.diversity_aware_rerank,
}

# Columns the metric functions need beyond the base ranked DF
FEATURE_COLS = ['video_id', 'y_relevant', 'implicit_completion_ratio',
                'item_age_days', 'author_id', 'tag']
available_feature_cols = [c for c in FEATURE_COLS if c in subset.columns]

all_results = []

# Group by user first, then by session within each user (to track prior session history)
for user_id, user_df in subset.groupby('user_id'):
    # Get chronological session order for this user
    session_order = (
        user_df.groupby('session_id')['time_ms'].min()
        .sort_values().index.tolist()
    )
    prior_video_ids = set()  # Accumulates across sessions for deja_vu_rate

    for session_id in session_order:
        session_df = user_df[user_df['session_id'] == session_id].copy()

        # Skip very small sessions (not meaningful to rank)
        if len(session_df) < 3:
            prior_video_ids.update(session_df['video_id'].tolist())
            continue

        for strategy_name, strategy_fn in STRATEGY_FUNCS.items():
            # Get ranked DataFrame from strategy
            ranked = strategy_fn(session_df)

            # Merge back feature columns needed for metrics
            features = session_df[available_feature_cols].drop_duplicates('video_id')
            ranked_with_features = ranked.merge(features, on='video_id', how='left')

            # Compute all metrics at once
            row = eval_metrics.score_all_metrics(
                ranked_with_features,
                strategy_name=strategy_name,
                prior_video_ids=prior_video_ids if prior_video_ids else None
            )
            row['user_id'] = user_id
            row['session_id'] = session_id
            row['session_size'] = len(session_df)
            all_results.append(row)

        prior_video_ids.update(session_df['video_id'].tolist())

results_df = pd.DataFrame(all_results)
print(f'Evaluation complete: {len(results_df)} rows ({results_df["session_id"].nunique()} sessions × 3 strategies)')
results_df.head()

Evaluation complete: 4506 rows (1502 sessions × 3 strategies)


,strategy,rel_mean_y_relevant,rel_mean_completion_ratio,rel_n_items,fresh_mean_item_age_days,fresh_median_item_age_days,fresh_fresh_item_ratio,fresh_n_items,div_unique_author_ratio,div_unique_tag_ratio,div_author_entropy,div_n_items,rep_consecutive_author_rate,rep_consecutive_tag_rate,rep_deja_vu_rate,rep_n_items,user_id,session_id,session_size
0,popularity_based,0.0000,0.0744,3,12.8955,13.2270,0.0000,3,1.0000,1.0000,1.0986,3,0.0000,0.0000,0.0000,3,5,5_2,3
1,freshness_boosted,0.0000,0.0744,3,12.8955,13.2270,0.0000,3,1.0000,1.0000,1.0986,3,0.0000,0.0000,0.0000,3,5,5_2,3
2,diversity_aware_rerank,0.0000,0.0744,3,12.8955,13.2270,0.0000,3,1.0000,1.0000,1.0986,3,0.0000,0.0000,0.0000,3,5,5_2,3
3,popularity_based,0.0000,0.0523,3,14.0986,13.4360,0.0000,3,1.0000,1.0000,1.0986,3,0.0000,0.0000,0.0000,3,5,5_4,3
4,freshness_boosted,0.0000,0.0523,3,14.0986,13.4360,0.0000,3,1.0000,1.0000,1.0986,3,0.0000,0.0000,0.0000,3,5,5_4,3


## 4. Aggregate Comparison Table

Mean metric values across all sessions, grouped by strategy.  
All metrics are defined so that **direction of improvement** matters:
- `rel_mean_y_relevant` → **higher is better**
- `fresh_mean_item_age_days` → **lower is better**
- `div_unique_author_ratio` → **higher is better**
- `rep_consecutive_author_rate` → **lower is better**

In [5]:
metric_cols = [
    'rel_mean_y_relevant',
    'rel_mean_completion_ratio',
    'fresh_mean_item_age_days',
    'fresh_fresh_item_ratio',
    'div_unique_author_ratio',
    'div_unique_tag_ratio',
    'div_author_entropy',
    'rep_consecutive_author_rate',
    'rep_consecutive_tag_rate',
    'rep_deja_vu_rate',
]
available_metric_cols = [c for c in metric_cols if c in results_df.columns]

comparison_table = (
    results_df.groupby('strategy')[available_metric_cols]
    .mean()
    .round(4)
)

# Reorder rows for readability
strategy_order = ['popularity_based', 'freshness_boosted', 'diversity_aware_rerank']
comparison_table = comparison_table.reindex(
    [s for s in strategy_order if s in comparison_table.index]
)

print('=== Strategy Comparison Table (mean across all sessions) ===')
comparison_table

=== Strategy Comparison Table (mean across all sessions) ===


,rel_mean_y_relevant,rel_mean_completion_ratio,fresh_mean_item_age_days,fresh_fresh_item_ratio,div_unique_author_ratio,div_unique_tag_ratio,div_author_entropy,rep_consecutive_author_rate,rep_consecutive_tag_rate,rep_deja_vu_rate
strategy,,,,,,,,,,
popularity_based,0.0480,0.1134,21.5075,0.0000,0.9999,0.9032,1.6811,0.0001,0.0481,0.0000
freshness_boosted,0.0480,0.1134,21.5075,0.0000,0.9999,0.9032,1.6811,0.0001,0.0483,0.0000
diversity_aware_rerank,0.0484,0.1145,21.5076,0.0000,0.9999,0.9048,1.6725,0.0001,0.0212,0.0000


In [6]:
# Save comparison table to outputs/
out_path = '../outputs/strategy_comparison.csv'
comparison_table.to_csv(out_path)
print(f'Saved comparison table to {out_path}')

# Also save the full per-session results for deeper analysis later
full_out_path = '../outputs/strategy_results_per_session.csv'
results_df.to_csv(full_out_path, index=False)
print(f'Saved per-session results to {full_out_path}')

Saved comparison table to ../outputs/strategy_comparison.csv
Saved per-session results to ../outputs/strategy_results_per_session.csv


## 5. Example Sessions — Side-by-Side Rankings

Pick one representative session and display the top-10 ranked items under each strategy.  
This makes the mechanical differences between strategies directly visible.

In [7]:
# Pick a session with a reasonable number of items for illustration
session_sizes = results_df[results_df['strategy'] == 'popularity_based'][['session_id', 'session_size']]
good_session_id = session_sizes[session_sizes['session_size'] >= 10]['session_id'].iloc[0]

example_session_df = subset[subset['session_id'] == good_session_id].copy()
print(f'Showing session: {good_session_id}  ({len(example_session_df)} items)')

display_cols = ['rank', 'video_id', 'score', 'author_id', 'tag']

features_ex = example_session_df[available_feature_cols].drop_duplicates('video_id')

for strategy_name, strategy_fn in STRATEGY_FUNCS.items():
    ranked = strategy_fn(example_session_df)
    ranked_full = ranked.merge(features_ex, on='video_id', how='left')
    print(f'\n--- {strategy_name} (top 10) ---')
    display_c = [c for c in ['rank', 'video_id', 'score', 'author_id', 'tag',
                              'y_relevant', 'item_age_days'] if c in ranked_full.columns]
    print(ranked_full[display_c].head(10).to_string(index=False))

Showing session: 5_9  (10 items)

--- popularity_based (top 10) ---
 rank  video_id  score  author_id   tag  y_relevant  item_age_days
    1      3704 0.5050    8662858    11           0        16.5781
    2      5183 0.3522      73132    39           0        17.5390
    3      5772 0.2139    8360153    39           0        17.5918
    4      5347 0.0952    6853179     6           0        15.5899
    5      4154 0.0877    1184266     6           0        16.5809
    6      4958 0.0766    7890584     3           0        15.5445
    7      2557 0.0305    3343900     3           0        17.5728
    8      4309 0.0160    3541832     5           0        17.5648
    9      3847 0.0000    3331825  8,67           0        16.5397
   10      4431 0.0000     287605 20,67           0        16.5430

--- freshness_boosted (top 10) ---
 rank  video_id  score  author_id   tag  y_relevant  item_age_days
    1      3704 0.2906    8662858    11           0        16.5781
    2      5183 0.1963   

## 6. Observations and Tradeoff Summary

### Popularity-Based
- Ranks items purely on engagement: explicit actions (like/follow/comment/forward) weighted 2×,
  implicit completion ratio filling in where explicit actions are absent.
- **Strength:** Items it surfaces have demonstrated user interest in this session.
- **Weakness:** Tends to cluster around a small set of high-engagement creators/tags.
  Repetition risk is highest here — the greedy engagement signal has no diversity incentive.
- **Expected scores:** Highest `mean_y_relevant`, highest `consecutive_author_rate`.

### Freshness-Boosted
- Applies an exponential decay multiplier to the popularity score based on item age.
  Decay time constant = 30 days (equivalent half-life ≈ 20.8 days).
- **Strength:** Newer content is surfaced earlier. Fresh items that also happen to be
  engaging score very highly.
- **Weakness:** Old but highly-engaging videos are suppressed, sometimes below less-relevant
  newer ones. Relevance score is expected to slightly decrease vs. popularity-based.
- **Tradeoff:** A deliberate freshness vs. relevance trade — tuning `decay_rate` shifts the balance.
- **Expected scores:** Lowest `mean_item_age_days` and highest `fresh_item_ratio`.

### Diversity-Aware Rerank
- Starts from the popularity-based pool and greedily reorders to suppress back-to-back author
  and tag repetition.
- **Strength:** Explicitly reduces `consecutive_author_rate` and `consecutive_tag_rate`.
  The user experiences more varied creators and topics in sequence.
- **Weakness:** Some engagement score is sacrificed: the greedy pick may skip the highest-scored
  item if it violates a recency window constraint.
- **Known limitation:** Session diversity only — does not account for cross-session deja-vu.
- **Expected scores:** Highest `unique_author_ratio`, lowest `consecutive_author_rate`.

### Quiet Assumptions (documented)
- `deja_vu_rate` is `NaN` for the very first session per user (no prior history exists).
- `tag = 'Unknown'` is treated as a real cluster — repeated `Unknown` items count as low diversity.
- `item_age_days = 0` for videos with missing `upload_dt` (imputed per `implementation_plan_v1`).
- All metrics are computed on the full session set (no fixed K), since session sizes here are small.

In [8]:
# Final sanity: confirm both output files exist
for fpath in [out_path, full_out_path]:
    size_kb = os.path.getsize(fpath) / 1024
    print(f'{fpath}: {size_kb:.1f} KB  OK')

../outputs/strategy_comparison.csv: 0.5 KB  OK
../outputs/strategy_results_per_session.csv: 672.4 KB  OK
